In [ ]:
!pip install ultralytics kaggle opencv-python matplotlib

In [ ]:
import os
# [TOKEN REMOVED]
# paste your new token after regenerating

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

# Write the token in the new format
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"Sifaul Islam","key":"KGAT_62106b6e88dad0807502fafe2d1f919f"}')

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Done!")

In [ ]:
!pip install kaggle -q
!kaggle datasets list

In [ ]:
!kaggle datasets download -d banuprasadb/visdrone-dataset
!unzip -q visdrone-dataset.zip -d visdrone/
!ls visdrone/

In [ ]:
import os

base = '/content/visdrone/VisDrone_Dataset'
for folder in os.listdir(base):
    path = os.path.join(base, folder)
    if os.path.isdir(path):  # only process folders, skip files
        print(f"📁 {folder}/")
        for sub in os.listdir(path):
            subpath = os.path.join(path, sub)
            if os.path.isdir(subpath):
                count = len(os.listdir(subpath))
                print(f"   └── {sub}/  ({count} files)")

In [ ]:
# Use these paths for everything going forward
img_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/images'
ann_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/annotations'

In [ ]:
import os

ann_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/labels'

sample_ann = os.path.join(ann_folder, sorted(os.listdir(ann_folder))[0])

print("Annotation file:", sample_ann)
print("\nFirst 10 lines:")
with open(sample_ann) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 9:
            break

In [ ]:
with open('/content/visdrone/VisDrone_Dataset/visdrone.yaml') as f:
    print(f.read())

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

img_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/images'
ann_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-train/labels'

CLASS_NAMES = {
    0: 'pedestrian',
    1: 'people',
    2: 'bicycle',
    3: 'car',
    4: 'van',
    5: 'truck',
    6: 'tricycle',
    7: 'awning-tricycle',
    8: 'bus',
    9: 'motor'
}

COLORS = {
    0: 'red',    # pedestrian
    1: 'orange', # people
    3: 'blue',   # car
    4: 'cyan',   # van
}

def visualize_sample(img_folder, ann_folder, index=0):
    img_files = sorted(os.listdir(img_folder))
    ann_files = sorted(os.listdir(ann_folder))

    img_path = os.path.join(img_folder, img_files[index])
    ann_path = os.path.join(ann_folder, ann_files[index])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(img)

    human_count = 0

    with open(ann_path) as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            x_c, y_c = float(parts[1]) * w, float(parts[2]) * h
            bw, bh = float(parts[3]) * w, float(parts[4]) * h

            # convert center format to top-left
            x1 = x_c - bw / 2
            y1 = y_c - bh / 2

            color = COLORS.get(cls, 'green')
            rect = patches.Rectangle((x1, y1), bw, bh,
                                      linewidth=1.5,
                                      edgecolor=color,
                                      facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, CLASS_NAMES.get(cls, str(cls)),
                    color=color, fontsize=6, fontweight='bold')

            if cls in [0, 1]:  # pedestrian + people
                human_count += 1

    ax.set_title(f"Sample Image — Total Humans: {human_count}", fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('sample_visualization.png', dpi=150)
    plt.show()
    print(f"Human count: {human_count}")

visualize_sample(img_folder, ann_folder, index=5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

img_files = sorted(os.listdir(img_folder))
ann_files = sorted(os.listdir(ann_folder))

for i, ax in enumerate(axes):
    img_path = os.path.join(img_folder, img_files[i * 10])  # spread out samples
    ann_path = os.path.join(ann_folder, ann_files[i * 10])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)

    human_count = 0
    car_count = 0

    with open(ann_path) as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            x_c, y_c = float(parts[1]) * w, float(parts[2]) * h
            bw, bh = float(parts[3]) * w, float(parts[4]) * h
            x1 = x_c - bw / 2
            y1 = y_c - bh / 2

            color = COLORS.get(cls, 'green')
            rect = patches.Rectangle((x1, y1), bw, bh,
                                      linewidth=1, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

            if cls in [0, 1]:
                human_count += 1
            if cls in [3, 4]:
                car_count += 1

    ax.set_title(f"Humans: {human_count}  |  Cars/Vans: {car_count}", fontsize=10)
    ax.axis('off')

plt.suptitle("VisDrone Dataset — Sample Images with Annotations", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('multi_sample_visualization.png', dpi=150)
plt.show()
print("Grid saved!")

In [ ]:
from collections import defaultdict

class_counts = defaultdict(int)
total_images = 0

for ann_file in sorted(os.listdir(ann_folder)):
    total_images += 1
    ann_path = os.path.join(ann_folder, ann_file)
    with open(ann_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 1:
                cls = int(parts[0])
                class_counts[CLASS_NAMES.get(cls, str(cls))] += 1

print(f"Total training images: {total_images}\n")
print("Object counts per class:")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (count // 500)
    print(f"  {cls:<20} {count:>6}  {bar}")

In [ ]:
!pip install ultralytics -q

In [ ]:
yaml_content = """
path: /content/visdrone/VisDrone_Dataset
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

nc: 10

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/content/visdrone_fixed.yaml', 'w') as f:
    f.write(yaml_content)

print("Fixed yaml created!")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/visdrone_fixed.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    name='visdrone_human_car',
    project='/content/runs',
    device=0
)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os

# Load your trained model
model = YOLO('/content/runs/visdrone_human_car-2/weights/best.pt')

# Pick a test image
img_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
test_images = sorted(os.listdir(img_folder))

def detect_and_count(img_path, conf=0.25):
    # Run inference
    results = model(img_path, conf=conf)[0]

    # Load image for display
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    human_count = 0
    car_count = 0

    for box in results.boxes:
        cls = int(box.cls)
        conf_score = float(box.conf)
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls in [0, 1]:  # pedestrian + people
            human_count += 1
            color = (255, 50, 50)   # red for humans
            label = f"Human {conf_score:.2f}"
        elif cls in [3, 4]:  # car + van
            car_count += 1
            color = (50, 50, 255)   # blue for cars
            label = f"Car {conf_score:.2f}"
        else:
            continue  # skip other classes

        # Draw box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    # Add count overlay on image
    overlay_text = f"Humans: {human_count}  |  Cars/Vans: {car_count}"
    cv2.rectangle(img, (0, 0), (350, 35), (0, 0, 0), -1)
    cv2.putText(img, overlay_text, (10, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.title(f"Detection Result — Humans: {human_count} | Cars/Vans: {car_count}", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('detection_result.png', dpi=150)
    plt.show()

    print(f"Humans detected: {human_count}")
    print(f"Cars/Vans detected: {car_count}")
    return human_count, car_count

# Test on a few images
for i in [5, 20, 50]:
    img_path = os.path.join(img_folder, test_images[i])
    print(f"\n--- Image: {test_images[i]} ---")
    detect_and_count(img_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

model_path = '/content/runs/visdrone_human_car-2/weights/best.pt'
model = YOLO(model_path)

img_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
test_images = sorted(os.listdir(img_folder))

# Pick 6 diverse images
selected = [5, 20, 50, 80, 120, 200]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, idx in enumerate(selected):
    img_path = os.path.join(img_folder, test_images[idx])
    results = model(img_path, conf=0.25, verbose=False)[0]

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    human_count = 0
    car_count = 0

    for box in results.boxes:
        cls = int(box.cls)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        if cls in [0, 1]:
            human_count += 1
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 50, 50), 2)
        elif cls in [3, 4]:
            car_count += 1
            cv2.rectangle(img, (x1, y1), (x2, y2), (50, 50, 255), 2)

    # Count overlay
    cv2.rectangle(img, (0, 0), (280, 30), (0, 0, 0), -1)
    cv2.putText(img, f"H:{human_count}  C:{car_count}", (8, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    axes[i].imshow(img)
    axes[i].set_title(f"Humans: {human_count} | Cars/Vans: {car_count}", fontsize=10)
    axes[i].axis('off')

plt.suptitle("YOLOv8n — Human & Car Detection Results on VisDrone Val Set", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('final_results_grid.png', dpi=150)
plt.show()
print("Results grid saved!")

In [ ]:
print("=" * 50)
print("   FINAL MODEL EVALUATION SUMMARY")
print("=" * 50)
print(f"  Model        : YOLOv8n (fine-tuned)")
print(f"  Dataset      : VisDrone 2019 Detection")
print(f"  Train images : 6,471")
print(f"  Val images   : 548")
print(f"  Epochs       : 20")
print(f"  Image size   : 640x640")
print("=" * 50)
print(f"  mAP@50       : 26.9%")
print(f"  mAP@50-95    : 14.9%")
print(f"  Precision    : 39.0%")
print(f"  Recall       : 30.2%")
print(f"  Inference    : ~2ms/image")
print("=" * 50)
print("  Per-class mAP@50:")
print(f"    car         : 69.9% ")
print(f"    bus         : 37.4% ")
print(f"    van         : 30.0%")
print(f"    pedestrian  : 27.6%")
print(f"    people      : 20.2%")
print(f"    motor       : 29.0%")
print(f"    bicycle     : 4.2%   (very small objects)")
print("=" * 50)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Create output folder in Drive
out = '/content/drive/MyDrive/Antlings_Assessment'
os.makedirs(out, exist_ok=True)

# Copy everything important
shutil.copy('sample_visualization.png', out)
shutil.copy('multi_sample_visualization.png', out)
shutil.copy('final_results_grid.png', out)
shutil.copy('/content/runs/visdrone_human_car-2/weights/best.pt', out)

# Copy training plots (auto-generated by ultralytics)
plots_src = '/content/runs/visdrone_human_car-2'
for f in os.listdir(plots_src):
    if f.endswith('.png'):
        shutil.copy(os.path.join(plots_src, f), out)

print("All files saved to Google Drive!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
path = '/content/drive/MyDrive/Antlings_Assessment'
if os.path.exists(path):
    print("Found! Files saved:")
    for f in os.listdir(path):
        print(f"  - {f}")
else:
    print("❌ Folder not found")

In [ ]:
import os, shutil
from ultralytics import YOLO

# Restore model
os.makedirs('/content/runs/visdrone_human_car-2/weights', exist_ok=True)
shutil.copy('/content/drive/MyDrive/Antlings_Assessment/best.pt',
            '/content/runs/visdrone_human_car-2/weights/best.pt')

# Test it loads
model = YOLO('/content/runs/visdrone_human_car-2/weights/best.pt')
print("Model restored and loaded!")

In [ ]:
# Re-download dataset if needed, or check if still there
import os
if os.path.exists('/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'):
    print("Dataset still exists!")
else:
    print("Dataset gone - need to redownload")
    !kaggle datasets download -d banuprasadb/visdrone-dataset
    !unzip -q visdrone-dataset.zip -d visdrone/

In [ ]:
import cv2
import os

img_folder = '/content/visdrone/VisDrone_Dataset/VisDrone2019-DET-val/images'
img_files = sorted(os.listdir(img_folder))

sequence = [f for f in img_files if f.startswith('0000001')]
print(f"Found {len(sequence)} images in sequence")

first = cv2.imread(os.path.join(img_folder, sequence[0]))
h, w = first.shape[:2]

out = cv2.VideoWriter('/content/test_video.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      10, (w, h))

for fname in sequence:
    img = cv2.imread(os.path.join(img_folder, fname))
    out.write(img)

out.release()
print(f"Video created with {len(sequence)} frames!")

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/visdrone_human_car-2/weights/best.pt')

results = model.track(
    source='/content/test_video.mp4',
    conf=0.25,
    tracker='bytetrack.yaml',
    save=True,
    project='/content/tracking_output',
    name='bytetrack_result',
    classes=[0, 1, 3, 4],
    verbose=False
)

print("Tracking complete!")

In [ ]:
import cv2
import matplotlib.pyplot as plt

tracked_video = '/content/tracking_output/bytetrack_result/test_video.mp4'
cap = cv2.VideoCapture(tracked_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total tracked frames: {total_frames}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, frame_num in enumerate([0, 5, 10, 15, 20, 25]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f"Frame {frame_num}", fontsize=10)
        axes[i].axis('off')

cap.release()
plt.suptitle("ByteTrack — Object Tracking on VisDrone Sequence", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('tracking_visualization.png', dpi=150)
plt.show()
print("Tracking visualization saved!")

In [ ]:
model = YOLO('/content/runs/visdrone_human_car-2/weights/best.pt')

human_ids = set()
car_ids = set()
frame_count = 0

for result in model.track(
    source='/content/test_video.mp4',
    conf=0.25,
    tracker='bytetrack.yaml',
    classes=[0, 1, 3, 4],
    stream=True,
    verbose=False
):
    frame_count += 1
    if result.boxes.id is not None:
        for box in result.boxes:
            tid = int(box.id)
            cls = int(box.cls)
            if cls in [0, 1]:
                human_ids.add(tid)
            elif cls in [3, 4]:
                car_ids.add(tid)

print("=" * 45)
print("   BYTETRACK TRACKING SUMMARY")
print("=" * 45)
print(f"  Frames processed      : {frame_count}")
print(f"  Unique humans tracked : {len(human_ids)}")
print(f"  Unique cars tracked   : {len(car_ids)}")
print(f"  Total unique objects  : {len(human_ids) + len(car_ids)}")
print("=" * 45)

In [ ]:
import os

base = '/content/tracking_output/bytetrack_result'
for f in os.listdir(base):
    print(f)

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Find the video file automatically
base = '/content/tracking_output/bytetrack_result'
video_file = [f for f in os.listdir(base) if f.endswith('.mp4')][0]
tracked_video = os.path.join(base, video_file)
print(f"Found: {tracked_video}")

cap = cv2.VideoCapture(tracked_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total frames: {total_frames}")

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i in range(min(9, total_frames)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f"Frame {i}", fontsize=10)
    axes[i].axis('off')

cap.release()
plt.suptitle("ByteTrack — Object Tracking on VisDrone Sequence", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('tracking_visualization.png', dpi=150)
plt.show()
print("Done")

In [ ]:
import os

base = '/content/tracking_output/bytetrack_result'
print("Files in tracking output:")
for f in os.listdir(base):
    print(f" - {f}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

tracked_video = '/content/tracking_output/bytetrack_result/test_video.avi'
cap = cv2.VideoCapture(tracked_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total frames: {total_frames}")

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i in range(min(9, total_frames)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[i].imshow(frame_rgb)
        axes[i].set_title(f"Frame {i}", fontsize=10)
    axes[i].axis('off')

cap.release()
plt.suptitle("ByteTrack — Object Tracking on VisDrone Sequence", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('tracking_visualization.png', dpi=150)
plt.show()
print("Tracking visualization saved!")

In [ ]:
# Convert avi to mp4 for demo video
!ffmpeg -i /content/tracking_output/bytetrack_result/test_video.avi \
        -vcodec libx264 /content/tracking_output/tracking_demo.mp4 -y -loglevel quiet
print("Converted to mp4!")

In [ ]:
import shutil

drive_path = '/content/drive/MyDrive/Antlings_Assessment'
shutil.copy('tracking_visualization.png', drive_path)
shutil.copy('/content/tracking_output/tracking_demo.mp4', drive_path)
print("Tracking files saved to Drive!")

In [ ]:
import shutil, os

drive_path = '/content/drive/MyDrive/Antlings_Assessment'

shutil.copy('tracking_visualization.png', drive_path)

# Convert avi to mp4 first
os.system('ffmpeg -i /content/tracking_output/bytetrack_result/test_video.avi /content/tracking_demo.mp4 -y -loglevel quiet')
shutil.copy('/content/tracking_demo.mp4', drive_path)

print("All tracking files saved to Drive!")
print("\nFiles in Drive now:")
for f in sorted(os.listdir(drive_path)):
    print(f"  - {f}")

In [ ]:
# Install git and configure
!git config --global user.email "sifaul1413@gmail.com"
!git config --global user.name "Sifaul-Islam"

# Clone your repo
!git clone https://github.com/Sifaul-Islam/antlings-drone-detection.git
%cd antlings-drone-detection

# Create outputs folder and copy files
import shutil, os
os.makedirs('outputs', exist_ok=True)

shutil.copy('/content/drive/MyDrive/Antlings_Assessment/sample_visualization.png', 'outputs/')
shutil.copy('/content/drive/MyDrive/Antlings_Assessment/multi_sample_visualization.png', 'outputs/')
shutil.copy('/content/drive/MyDrive/Antlings_Assessment/final_results_grid.png', 'outputs/')
shutil.copy('/content/drive/MyDrive/Antlings_Assessment/tracking_visualization.png', 'outputs/')

print("Files copied!")
!ls outputs/

In [ ]:
GITHUB_USERNAME = "Sifaul-Islam"
# [TOKEN REMOVED]
REPO_NAME = "antlings-drone-detection"

# [TOKEN REMOVED]
!git push origin main

In [ ]:
!git remote -v

In [ ]:
!git config user.name
!git config user.email

In [ ]:
GITHUB_USERNAME = "Sifaul-Islam"
# [TOKEN REMOVED]
REPO_NAME = "antlings-drone-detection"

!git remote remove origin
# [TOKEN REMOVED]
!git push -u origin main --force

In [ ]:
import shutil

# Copy notebook to repo folder
# First check if notebook exists in drive
import os
drive_path = '/content/drive/MyDrive/Antlings_Assessment'
print("Files in Drive:")
for f in os.listdir(drive_path):
    print(f"  - {f}")

In [ ]:
import shutil, os

repo = '/content/antlings-drone-detection'

# Copy notebook
shutil.copy('/content/drive/MyDrive/Antlings_Assessment/task.ipynb', repo)

# Copy training plots to outputs folder
plots = ['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png',
         'BoxR_curve.png', 'confusion_matrix_normalized.png',
         'confusion_matrix.png', 'results.png']

for f in plots:
    shutil.copy(f'/content/drive/MyDrive/Antlings_Assessment/{f}',
                f'{repo}/outputs/')

print("All files copied!")
print("\nRepo contents:")
for f in os.listdir(repo):
    print(f"  - {f}")
print("\nOutputs folder:")
for f in os.listdir(f'{repo}/outputs'):
    print(f"  - {f}")

In [ ]:
%cd /content/antlings-drone-detection

!git add .
!git commit -m "Add notebook and training plots"
!git push origin main

In [ ]:
GITHUB_USERNAME = "Sifaul-Islam"
# [TOKEN REMOVED]
REPO_NAME = "antlings-drone-detection"

# Remove the nested repo problem
!git rm --cached antlings-drone-detection -r 2>/dev/null || true

# Push with token
# [TOKEN REMOVED]


In [ ]:
GITHUB_USERNAME = "Sifaul-Islam"
# [TOKEN REMOVED]
REPO_NAME = "antlings-drone-detection"

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
import os
os.chdir('/content')
# [TOKEN REMOVED]
os.chdir(f'/content/{REPO_NAME}')

!git config user.email "sifaul1413@gmail.com"
!git config user.name "Sifaul-Islam"

print("Repo cloned!")
!ls